# Device Access — Two Partitions in One Flat Table

ACS table `B28001` lists eleven numbers about computing devices in a household. Four of them
name a device type: desktop or laptop, smartphone, tablet, other. Adding those four together is
the obvious thing to do, and it produces a number **148.2% larger than the count of households
that own anything at all** — in every one of **3,222 counties**, without exception.

The reason is that four other rows in the same list end with the words *"with no other type of
computing device"*. Those four are mutually exclusive; the first four are not. **The API marks
the difference nowhere.** `B28001_005E` and `B28001_006E` are adjacent integers of the same type
in the same response, and one may be summed with its siblings while the other may not.

This notebook separates the two partitions, tests both identities, and then uses the one that is
real. Of households owning any device, **only 14.0% own exactly one**. The single largest
single-device group is the phone: **12,255,482 households — 9.52% of all households** — whose
only computing device is a smartphone.

That is a measure D16 cannot see. D16 is built on `B28002` and measures whether a household
*subscribes* to internet; **89.59%** have broadband. Subscription and device ownership move
together strongly (ρ = **−0.8509** against the no-device share) and they are not the same thing:
among the 806 counties in the **top quartile of broadband subscription**, the smartphone-only
share still runs from 1.46% to **36.28%**.

## Learning Objectives

1. **Read variable labels before summing anything**, and recognise when one flat list contains
   two different partitions.
2. **Test every identity a table implies**, not only the one you expect to hold.
3. **Quantify a double-count** rather than noting that one exists.
4. **Distinguish a subscription from a capability**, and see that a strong correlation between
   them does not make them interchangeable.
5. **Know when to cite a settled result instead of recomputing it.**

## Prerequisites

- Handbook §3 (denominators, universes and partitions) — this notebook is that section's
  clearest single case.
- Handbook §1.1 (ACS margins of error), and the ratio formula for a share whose numerator is a
  subset of its denominator.
- D16 (internet subscription, `B28002`), whose open half this notebook closes.
- Handbook §7.3 — cited in section 8, not redone.

## Data Sources

**U.S. Census Bureau, ACS 5-year 2023**, two tables at county level:

- **`B28001`** — *Types of Computing Devices in Household.* Eleven estimates, of which four
  overlap and four do not.
- **`B28002`** — *Presence and Types of Internet Subscriptions in Household.* The table D16 is
  built on, used here only in section 7.

Puerto Rico municipios are county equivalents and are **kept**, because a device-access question
about the United States that drops them is answering a different question. They dominate the
smartphone-only ranking, so section 6 reports it both ways.

## 1. Setup

The two row groups are named explicitly in configuration rather than derived at the point of
use, because which rows may be summed is the substance of this notebook and not an
implementation detail.

In [1]:
"""Configuration."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.sources import fetch_acs_wide
from krl_tutorials_data.uncertainty import Estimate, Z90, ratio_moe

YEAR = 2023

#: B28001 -- types of computing device. Labels verbatim from Census metadata,
#: because the whole notebook turns on reading them.
DEVICES = {
    "B28001_001E": "Total households",
    "B28001_001M": "Total households, MOE",
    "B28001_002E": "Has one or more types of computing devices",
    "B28001_002M": "Has one or more types of computing devices, MOE",
    "B28001_003E": "Has ... Desktop or laptop",
    "B28001_004E": "Has ... Desktop or laptop with no other type of computing device",
    "B28001_005E": "Has ... Smartphone",
    "B28001_006E": "Has ... Smartphone with no other type of computing device",
    "B28001_006M": "Smartphone-only, MOE",
    "B28001_007E": "Has ... Tablet or other portable wireless computer",
    "B28001_008E": "Has ... Tablet or other portable wireless computer with no other type",
    "B28001_009E": "Has ... Other computer",
    "B28001_010E": "Has ... Other computer with no other type of computing device",
    "B28001_011E": "No Computer",
    "B28001_011M": "No Computer, MOE",
}

#: B28002 -- internet subscription. D16 is built on this table; section 6 joins
#: the two to separate a subscription from a capability.
INTERNET = {
    "B28002_001E": "Total households (internet table)",
    "B28002_004E": "With broadband of any type",
    "B28002_004M": "With broadband of any type, MOE",
    "B28002_013E": "No Internet access",
}

#: The four device TYPES. A household with a laptop and a phone is in two of
#: them, so they are not a partition and must not be summed.
TYPE_ROWS = ["B28001_003E", "B28001_005E", "B28001_007E", "B28001_009E"]

#: The four "with no other type" rows. These ARE mutually exclusive.
ONLY_ROWS = ["B28001_004E", "B28001_006E", "B28001_008E", "B28001_010E"]

pd.set_option("display.width", 112)
print(f"vintage : ACS 5-year {YEAR}, county level")
print(f"overlapping device types : {', '.join(TYPE_ROWS)}")
print(f"mutually exclusive rows  : {', '.join(ONLY_ROWS)}")


vintage : ACS 5-year 2023, county level
overlapping device types : B28001_003E, B28001_005E, B28001_007E, B28001_009E
mutually exclusive rows  : B28001_004E, B28001_006E, B28001_008E, B28001_010E


In [2]:
"""Retrieve both tables."""
devices, device_prov = fetch_acs_wide(year=YEAR, variables=DEVICES, geography="county:*")
internet, internet_prov = fetch_acs_wide(year=YEAR, variables=INTERNET, geography="county:*")
devices = devices.dropna(subset=["B28001_001E"]).reset_index(drop=True)
internet = internet.dropna(subset=["B28002_001E"]).reset_index(drop=True)
counties = devices.merge(internet.drop(columns=["NAME"]), on=["state", "county"], how="inner")

puerto_rico = counties["NAME"].str.contains("Puerto Rico")
print(f"counties and equivalents : {len(counties):,}")
print(f"  of which Puerto Rico   : {int(puerto_rico.sum())}")
print(f"households represented   : {int(counties['B28001_001E'].sum()):,}")
print(f"payload sha256 (B28001)  : {device_prov.content_sha256[:16]}...")
print(f"payload sha256 (B28002)  : {internet_prov.content_sha256[:16]}...")
print()
print("Puerto Rico municipios are county equivalents and are kept. Section 5 reports")
print("the smartphone-only ranking with and without them, because they dominate it.")


counties and equivalents : 3,222
  of which Puerto Rico   : 78
households represented   : 128,722,108
payload sha256 (B28001)  : 5a1efd54635518da...
payload sha256 (B28002)  : 0b87f5fb6e75e037...

Puerto Rico municipios are county equivalents and are kept. Section 5 reports
the smartphone-only ranking with and without them, because they dominate it.


## 2. The distinction lives in the words

Print the labels and the difference is visible. Nothing else makes it visible.

In [3]:
"""Two partitions, one flat list, distinguished only by the label text."""
labels = pd.DataFrame({
    "variable": list(DEVICES),
    "label": list(DEVICES.values()),
})
estimates = labels[labels["variable"].str.endswith("E")].copy()
estimates["kind"] = np.where(
    estimates["variable"].isin(TYPE_ROWS), "device type (OVERLAPPING)",
    np.where(estimates["variable"].isin(ONLY_ROWS), "single-device (exclusive)", "margin"),
)
print(estimates.to_string(index=False))
print()
print("Nothing in the API response marks the difference. `B28001_005E` and")
print("`B28001_006E` are adjacent integers of the same type, and one may be summed")
print("with its siblings while the other may not. The distinction exists only in")
print("the words 'with no other type of computing device'.")


   variable                                                                 label                      kind
B28001_001E                                                      Total households                    margin
B28001_002E                            Has one or more types of computing devices                    margin
B28001_003E                                             Has ... Desktop or laptop device type (OVERLAPPING)
B28001_004E      Has ... Desktop or laptop with no other type of computing device single-device (exclusive)
B28001_005E                                                    Has ... Smartphone device type (OVERLAPPING)
B28001_006E             Has ... Smartphone with no other type of computing device single-device (exclusive)
B28001_007E                    Has ... Tablet or other portable wireless computer device type (OVERLAPPING)
B28001_008E Has ... Tablet or other portable wireless computer with no other type single-device (exclusive)
B28001_009E                 

Four rows name a device type and four name a device type *"with no other type of
computing device"*. The second phrase turns an overlapping category into an exclusive one.

Everything else about these eight numbers is identical: same table, same geography, same units,
adjacent variable codes, same JSON type. A client that selects variables by pattern — every
`B28001_*E`, or every row under "Has one or more types" — gets both groups and no way to tell
them apart.

**[rule]** Carry the published label alongside every variable code, into provenance, and read it
before aggregating. A code is an address, not a definition.

## 3. The identity that holds

In [4]:
"""The one identity that does hold."""
total = counties["B28001_001E"]
has_device = counties["B28001_002E"]
no_computer = counties["B28001_011E"]
residual = has_device + no_computer - total

print(f"has-device + no-computer == total, exactly in "
      f"{int((residual == 0).sum()):,} of {len(counties):,} counties")
print(f"largest absolute discrepancy anywhere: {int(residual.abs().max())}")
print()
print(f"national households        : {int(total.sum()):,}")
print(f"  with a computing device  : {int(has_device.sum()):,} "
      f"({100 * has_device.sum() / total.sum():.2f}%)")
print(f"  with none                : {int(no_computer.sum()):,} "
      f"({100 * no_computer.sum() / total.sum():.2f}%)")


has-device + no-computer == total, exactly in 3,222 of 3,222 counties
largest absolute discrepancy anywhere: 0

national households        : 128,722,108
  with a computing device  : 121,880,959 (94.69%)
  with none                : 6,841,149 (5.31%)


**Exact in 3,222 of 3,222 counties**, with a largest discrepancy of **0**. Households
with a device plus households without one equals all households, everywhere, to the unit.

This is the identity most analysts would test if they tested one, and it passing is what makes
the next section dangerous: the table looks internally consistent because in this respect it
is.

## 4. The identity that does not

Now sum the four device types and compare against the same denominator.

In [5]:
"""And the one that does not."""
type_sum = counties[TYPE_ROWS].sum(axis=1)
excess = type_sum - has_device
share = 100 * excess / has_device

print(f"sum of the four device types == has-device, exactly in "
      f"{int((excess == 0).sum())} of {len(counties):,} counties")
print(f"the sum EXCEEDS has-device in {int((excess > 0).sum()):,} counties")
print()
print(f"national: types sum to {int(type_sum.sum()):,} against "
      f"{int(has_device.sum()):,} households with a device")
print(f"          an overcount of {100 * type_sum.sum() / has_device.sum() - 100:.1f}%")
print()
print(f"per county, the excess as a share of has-device:")
print(f"  median {share.median():.1f}%   minimum {share.min():.1f}%   maximum {share.max():.1f}%")
print()
print("A household with a laptop and a phone is counted in both rows. Summing them")
print("does not double-count by a little; it produces a figure two and a half times")
print("the number of households that own anything.")


sum of the four device types == has-device, exactly in 0 of 3,222 counties
the sum EXCEEDS has-device in 3,222 counties

national: types sum to 302,501,089 against 121,880,959 households with a device
          an overcount of 148.2%

per county, the excess as a share of has-device:
  median 133.6%   minimum 49.0%   maximum 182.4%

A household with a laptop and a phone is counted in both rows. Summing them
does not double-count by a little; it produces a figure two and a half times
the number of households that own anything.


**Exact in 0 of 3,222. The sum exceeds the count of device-owning households in every
single county.**

Nationally the four types sum to **302,501,089** against **121,880,959** households that own
anything — an overcount of **148.2%**. Per county the excess as a share of the device-owning
count has a median of **133.6%** and runs from **49.0%** to **182.4%**: even in the county where
the overlap is smallest, the naive sum is half again too large.

This is not a rounding artifact or an edge case. A household with a laptop and a phone is
counted in both rows, as it should be, because the question asked was *which of these do you
have* and the answer can be several. The rows are correct. The sum is meaningless.

**[rule]** A table that passes one identity has not been validated. Test every identity the
structure implies, including the ones you expect to fail — the failure is where the structure is.

## 5. The rows that are a partition

The four *"with no other type"* rows are mutually exclusive by construction. They should sum to
at most the device-owning count, and the remainder is households with more than one kind of
device.

In [6]:
"""The rows that are a partition, and what they say."""
only_sum = counties[ONLY_ROWS].sum(axis=1)
print(f"single-device sum <= has-device, in {int((only_sum <= has_device).sum()):,} "
      f"of {len(counties):,} counties")
print()
national = pd.DataFrame({
    "households": [counties[c].sum() for c in ONLY_ROWS],
    "label": [DEVICES[c].replace("Has ... ", "") for c in ONLY_ROWS],
}, index=ONLY_ROWS)
national["share of device-owning"] = 100 * national["households"] / has_device.sum()
print(national.to_string(formatters={
    "households": "{:,.0f}".format, "share of device-owning": "{:.2f}".format}))
print()
print(f"single-device households : {int(only_sum.sum()):,} "
      f"({100 * only_sum.sum() / has_device.sum():.1f}% of those owning anything)")
print(f"multi-device households  : {100 * (1 - only_sum.sum() / has_device.sum()):.1f}%")


single-device sum <= has-device, in 3,222 of 3,222 counties

            households                                                          label share of device-owning
B28001_004E  3,720,009       Desktop or laptop with no other type of computing device                   3.05
B28001_006E 12,255,482              Smartphone with no other type of computing device                  10.06
B28001_008E  1,003,789  Tablet or other portable wireless computer with no other type                   0.82
B28001_010E     33,799          Other computer with no other type of computing device                   0.03

single-device households : 17,013,079 (14.0% of those owning anything)
multi-device households  : 86.0%


**Single-device households are 17,013,079 — 14.0% of households owning anything.**
The other **86.0%** own more than one kind of device.

Within that 14%, the phone dominates: **12,255,482 households**, **10.06%** of device-owning
households, against **3.05%** for desktop-or-laptop-only and **0.82%** for tablet-only. The
"other computer only" row is **33,799** households nationally, **0.03%** — small enough that any
county-level statement about it will be mostly margin of error.

**[rule]** When a table offers an overlapping breakdown and an exclusive one, the exclusive one
is the partition. Use it for anything that sums, and use the overlapping one only for
per-category rates against the full denominator.

## 6. Smartphone-only

Of all the numbers in this table this is the one worth carrying somewhere else. It is the
population that owns a computing device, has internet on it, and cannot do most of what a
computer does.

In [7]:
"""Smartphone-only: the measure a subscription table cannot see."""
smartphone_only = counties["B28001_006E"] / counties["B28001_001E"]
national_share = counties["B28001_006E"].sum() / counties["B28001_001E"].sum()

print(f"national smartphone-only share of households : {100 * national_share:.2f}%")
print(f"county median                                : {100 * smartphone_only.median():.2f}%")
print(f"county range                                 : {100 * smartphone_only.min():.2f}% "
      f"to {100 * smartphone_only.max():.2f}%")
print()
ranked = counties.assign(share=smartphone_only)
for label, subset in (("all counties", ranked),
                      ("excluding Puerto Rico", ranked[~puerto_rico])):
    top = subset.nlargest(5, "share")[["NAME", "B28001_001E", "B28001_006E", "share"]]
    print(f"highest smartphone-only share, {label}:")
    print(top.to_string(index=False, formatters={
        "B28001_001E": "{:,.0f}".format, "B28001_006E": "{:,.0f}".format,
        "share": "{:.1%}".format}))
    print()


national smartphone-only share of households : 9.52%
county median                                : 12.23%
county range                                 : 0.00% to 41.95%

highest smartphone-only share, all counties:
                          NAME B28001_001E B28001_006E share
Vieques Municipio, Puerto Rico       2,663       1,117 41.9%
 Manatí Municipio, Puerto Rico      14,813       5,782 39.0%
  Kusilvak Census Area, Alaska       1,891         732 38.7%
 Cataño Municipio, Puerto Rico       8,998       3,430 38.1%
  Loíza Municipio, Puerto Rico       8,730       3,220 36.9%

highest smartphone-only share, excluding Puerto Rico:
                        NAME B28001_001E B28001_006E share
Kusilvak Census Area, Alaska       1,891         732 38.7%
  Tunica County, Mississippi       3,600       1,307 36.3%
        Newton County, Texas       4,898       1,777 36.3%
     Culberson County, Texas         702         253 36.0%
        Brooks County, Texas       2,884       1,014 35.2%



**9.52% of all U.S. households have a smartphone and no other computing device** —
note the denominator; it is 10.06% of device-owning households and 9.52% of all households, and
the two get confused easily.

The county median is **12.23%**, higher than the national share because the national figure is
household-weighted and the counties with the highest shares are small. The range runs from
**0.00%** to **41.95%**.

The top of the ranking is dominated by Puerto Rico — Vieques at **41.9%**, Manatí at 39.0% —
which is why section 1 keeps the municipios and this section reports the list twice. Excluding
them the highest is **Kusilvak Census Area, Alaska at 38.7%**, followed by Tunica County,
Mississippi and Newton County, Texas at 36.3%.

A national figure under ten percent and county figures approaching forty is the shape of this
measure, and either alone is misleading.

## 7. A subscription is not a capability

D16 answers whether a household subscribes to internet. This table answers what it can use the
subscription on. The two are closely related, which is exactly why they get substituted for one
another.

In [8]:
"""A subscription is not a capability."""
broadband = counties["B28002_004E"] / counties["B28002_001E"]
no_device = no_computer / total

print(f"national broadband subscription share : "
      f"{100 * counties['B28002_004E'].sum() / counties['B28002_001E'].sum():.2f}%")
print(f"national no-computing-device share    : {100 * no_computer.sum() / total.sum():.2f}%")
rho, pval = stats.spearmanr(broadband, no_device)
print(f"Spearman(broadband share, no-device share) = {rho:+.4f}  p = {pval:.3g}  "
      f"n = {len(counties):,}")
print()
best = counties.assign(broadband=broadband, share=smartphone_only)
best = best[best["broadband"] > best["broadband"].quantile(0.75)]
print(f"among the {len(best):,} counties in the top quartile of broadband subscription:")
print(f"  smartphone-only share: median {100 * best['share'].median():.2f}%, "
      f"range {100 * best['share'].min():.2f}% to {100 * best['share'].max():.2f}%")
print()
print("Broadband subscription and device ownership move together strongly. They are")
print("not the same thing, and the gap does not close in the best-connected places.")


national broadband subscription share : 89.59%
national no-computing-device share    : 5.31%
Spearman(broadband share, no-device share) = -0.8509  p = 0  n = 3,222

among the 806 counties in the top quartile of broadband subscription:
  smartphone-only share: median 7.96%, range 1.46% to 36.28%

Broadband subscription and device ownership move together strongly. They are
not the same thing, and the gap does not close in the best-connected places.


**89.59% of households subscribe to broadband of some type; 5.31% own no computing
device at all.** The two move together strongly — Spearman **−0.8509** between broadband share
and no-device share across **3,222** counties, with a p-value indistinguishable from zero.

A correlation that strong is usually the end of an argument. Here it is not, because the
question is not whether the two agree on average but whether one can stand in for the other in a
particular place. Among the **806 counties in the top quartile of broadband subscription**, the
smartphone-only share has a median of **7.96%** and a range of **1.46% to 36.28%**.

So in the best-connected quarter of American counties there is still a county where more than a
third of households have only a phone. A programme targeting places by subscription rate would
not find it.

**[rule]** A strong correlation between two measures licenses using one to predict the other's
average. It does not license substituting one for the other in a decision about a particular
unit.

## 8. What the margins permit

Handbook §7.3 has settled what a 3,222-county table licenses, across four notebooks. This
section reports the two numbers needed to place this measure in that result and does not redo
it.

In [9]:
"""What the margins permit, without redoing handbook 7.3."""
value = smartphone_only.to_numpy(float)
moe = np.array([
    ratio_moe(
        Estimate(value=counties.loc[i, "B28001_006E"], moe=counties.loc[i, "B28001_006M"]),
        Estimate(value=counties.loc[i, "B28001_001E"], moe=counties.loc[i, "B28001_001M"]),
    )
    for i in counties.index
])
error = moe / Z90
spread = value.max() - value.min()

print(f"counties                              : {len(value):,}")
print(f"between-county spread                 : {spread:.4f}")
print(f"median 95% interval width             : {2 * 1.96 * np.median(error):.4f}")
print(f"interval width as a share of spread   : "
      f"{2 * 1.96 * np.median(error) / spread:.4f}")
print()
national_value = float(national_share)
detached = np.abs(value - national_value) / error > 1.96
print(f"counties separating from the national {100 * national_value:.2f}% at 95%: "
      f"{int(detached.sum()):,} ({detached.mean():.1%})")
print()
print("Handbook 7.3 settles what a table of 3,222 counties licenses: detection")
print("against a benchmark, not an ordering. That result is not recomputed here.")


counties                              : 3,222
between-county spread                 : 0.4195
median 95% interval width             : 0.0567
interval width as a share of spread   : 0.1353

counties separating from the national 9.52% at 95%: 2,051 (63.7%)

Handbook 7.3 settles what a table of 3,222 counties licenses: detection
against a benchmark, not an ordering. That result is not recomputed here.


The 95% interval is **13.53%** of the between-county spread — comparable to D33's
10.78% and far tighter than D10's 0.89 — and **2,051 counties (63.7%)** separate from the
national 9.52% at 95%.

§7.3's conclusion applies unchanged: detection against a benchmark works, ordering does not, and
at 3,222 units the number of comparisons a rank must win is the binding constraint rather than
the precision of any one estimate. **Naming which counties differ from the national figure is
supported. Publishing a ranked list of 3,222 is not.**

## Limitations

1. **Ownership is not access, and access is not use.** The table records what a household
   reports having. It says nothing about whether the device works, is shared among several
   people, or is used.

2. **"Smartphone-only" is a device statement, not a capability measurement.** The inference that
   such households can do less is reasonable and is not measured anywhere here.

3. **The overlapping rows are correct and only their sum is wrong.** Nothing in section 4 is a
   criticism of the Census Bureau; the table answers a multiple-response question and reports it
   accordingly.

4. **`B28001_010E` is too small to use at county level.** 33,799 households nationally, 0.03% of
   device owners. Any county statement about it is mostly margin.

5. **The county median and the national share are different quantities** (12.23% against 9.52%)
   and the difference is county size, not a discrepancy.

6. **Section 7 is a correlation across counties.** It supports no claim about which households
   within a county lack devices, and the ecological inference is not made.

7. **One vintage, no time series.** ACS 5-year 2023 only; nothing here shows whether the
   smartphone-only share is rising or falling.

8. **Puerto Rico is included throughout except where section 6 states otherwise.**

## Key Takeaways

1. **One flat list, two partitions.** Four `B28001` rows overlap and four do not, and only the
   label text says which.

2. **Summing the overlapping rows overcounts by 148.2%**, in 3,222 of 3,222 counties.

3. **Test every identity, not the one you expect to hold.** `has-device + no-computer == total`
   is exact everywhere; that is what makes the other failure easy to miss.

4. **86.0% of device-owning households own more than one kind**, so the exclusive rows cover
   only 14.0% and are a small slice, not a breakdown of the whole.

5. **12,255,482 households — 9.52% of all households — have only a smartphone**, county median
   12.23%, maximum 41.95%.

6. **Subscription is not capability.** ρ = −0.8509, and in the top broadband quartile the
   smartphone-only share still reaches 36.28%.

7. **Cite a settled result rather than recomputing it.** §7.3 already answers what 3,222 counties
   license; this notebook supplies its two numbers and stops.

## Exercises

1. **Build the per-type rates correctly.** Section 4 shows the types cannot be summed. Compute
   each type's share of all households instead, state why that is legitimate where the sum is
   not, and say what the four shares do and do not add up to.

2. **Find the other multiple-response tables.** `B28001` is not unique in ACS. Search the 2023
   variable metadata for other tables whose labels contain "with no other" or similar, and report
   how many have the same two-partition structure.

3. **Close the ecological gap, or show it cannot be closed.** Section 7 correlates county shares.
   State precisely what household-level claim would follow, what data would be needed, and
   whether ACS can supply it.

4. **Rank properly.** Using `rank_intervals`, compute how many of the 3,222 counties have a
   determined rank on smartphone-only share, and compare the answer to §7.3's four cases.

5. **Test the small cell.** Limitation 4 asserts `B28001_010E` is unusable at county level.
   Verify it: compute the coefficient of variation per county and report what share fail the NCHS
   reliability screen.

6. **Write the targeting rule.** A programme has budget for 200 counties. Draft the selection
   rule you would defend, using both tables, and state what it will systematically miss.

## References

- U.S. Census Bureau. *ACS table B28001: Types of Computing Devices in Household.*
  https://data.census.gov/table/ACSDT5Y2023.B28001
- U.S. Census Bureau. *ACS table B28002: Presence and Types of Internet Subscriptions in
  Household.* https://data.census.gov/table/ACSDT5Y2023.B28002
- U.S. Census Bureau. *Understanding and Using American Community Survey Data* — multiple-response
  tables, and the proportion and ratio margin-of-error formulas.
  https://www.census.gov/programs-surveys/acs/library/handbooks/general.html
- Handbook §1.1 (margins of error), §3 (denominators, universes and partitions), §7.3 (precision
  and rank).
- D16 (`06-infrastructure-environment/D16_internet_and_technology_access`), whose subscription
  coverage this notebook complements.